In [0]:
import dlt
from pyspark.sql.functions import col

# 1. Definimos la variable con la ruta física de los JSONs en tu Volume
json_path = "/Volumes/main/default/bronze"

# 2. Declaramos la Streaming Table de destino
@dlt.table(
    name="table_destino",
    comment="Tabla de ingesta bronze con Auto Loader y reglas de calidad aplicadas.",

    # 💡 AQUÍ SE INCORPORAN LAS PROPERTIES
    tblproperties={
        "delta.enableChangeDataFeed": "true",           # Habilita el feed de cambios Delta (CDF)
        "pipelines.metastore.tableName": "table_destino",
        "my_custom_metadata.layer": "bronze",            # Puedes añadir tags personalizados de auditoría
        "delta.logRetentionDuration": "interval 7 days",
        "delta.deletedFileRetentionDuration": "interval 7 days"
    }
)
# Aplicamos las expectativas equivalentes a tu filtro clásico de Spark
@dlt.expect_or_drop("id_no_nulo", "id IS NOT NULL")              # Elimina filas malas del stream [cite: 111, 263]
@dlt.expect_or_drop("edad_rango_logico", "age >= 0 AND age < 120") # Elimina filas con edad inválida [cite: 111]
def table_destino():
    # Retornamos la lectura declarativa con Auto Loader (cloudFiles)
    # NOTA: Aquí NO declaramos schemaLocation ni checkpointLocation 
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaHints", "id INT, name STRING, age INT") # Mantenemos tus hints
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")       # Mantenemos evolución [cite: 10]
        .option("cloudFiles.maxFilesPerTrigger", 10)
        .option("cloudFiles.maxBytesPerTrigger", "10m")
        .option("cloudFiles.useNotifications", "false")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(json_path)
    )